# Day 1 - Exercise 01: Understanding GenAI vs Agentic AI

## Real-World Scenario: Customer Service Bot

**Context:** You work at an e-commerce company. The support team receives hundreds of queries daily about order status, returns, and product information.

In this exercise, you'll build TWO versions of a customer service assistant:
1. **GenAI Version** — Simple Q&A (no tools, no memory)
2. **Agentic AI Version** — Can look up orders, check inventory, and remember conversation context

By the end, you'll clearly see the difference between reactive GenAI and proactive Agentic AI.

---

### SETUP

In [2]:
%pip install python-dotenv langchain-anthropic langchain-core langchain --break-system-packages

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.6/133.6 kB 2.9 MB/s eta 0:00:0000:01
Note: you may need to restart the kernel to use updated packages.


In [8]:
from dotenv import load_dotenv
import os
load_dotenv()

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
import os
# Initialize Claude
llm = ChatAnthropic(
    api_key=os.environ.get("KEY"),
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    base_url="https://llmgw-wp.tekstac.com"
)
print("Setup complete!")

Setup complete!


---
## Part 1: GenAI Version (Reactive, No Tools, No Memory)

This version can only answer based on what it knows. It cannot:
- Look up actual order information
- Remember what the customer said earlier
- Take any real actions

In [9]:
# GenAI Customer Service Bot - Simple Q&A

def genai_support_bot(question: str) -> str:
    """Simple GenAI bot - no memory, no tools, just Q&A"""
    messages = [
        SystemMessage(content="You are a helpful customer service assistant for ShopEasy e-commerce."),
        HumanMessage(content=question)
    ]
    response = llm.invoke(messages)
    return response.content

# Test the GenAI bot
print("=" * 60)
print("GenAI Bot Response:")
print("=" * 60)
genai_support_bot("What is the status of my order #12345?")

GenAI Bot Response:


"I'd be happy to help you check your order status! However, I don't have access to ShopEasy's order database or your account information.\n\nTo find the status of order #12345, you can:\n\n1. **Check your email** - Look for an order confirmation or shipping notification email\n2. **Log into your account** - Visit the ShopEasy website and sign in to view your order history and tracking details\n3. **Contact customer service directly** - Reach out to our support team with your order number, and they can pull up the details for you\n\nIf you're having trouble accessing your account or need immediate assistance, please let me know and I can help guide you to the right support channel. Is there anything else I can help with regarding your order?"

###  Problem: The GenAI bot CANNOT actually look up order #12345!

It can only give generic responses. Let's see another limitation...

In [10]:
# Testing memory limitation
print("Turn 1:")
print(genai_support_bot("My name is Sarah and my order number is #12345"))

print("\n" + "=" * 60)
print("Turn 2:")
print(genai_support_bot("What is my name and order number?"))

Turn 1:
Hello Sarah! 👋 

Thank you for contacting ShopEasy! I have your order number **#12345** on file.

How can I help you today? I'm happy to assist with:

- **Order status** - Where your package is and when it'll arrive
- **Returns or exchanges** - If you need to send something back
- **Product questions** - Details about items in your order
- **Billing or account issues** - Payment or account concerns
- **General support** - Any other questions

What can I do for you?

Turn 2:
I don't have access to any of your personal information, account details, or order history. I'm a general customer service assistant without access to customer databases or accounts.

To help you with your order, I'd need you to provide:
- Your name
- Your order number
- Any other relevant details about your inquiry

Once you share those details, I'll be happy to assist you with questions about your order, shipping, returns, or other customer service needs!


###  Problem: It forgot everything from Turn 1!

Each call is independent — no memory between turns.

---

## Part 2: Agentic AI Version (Tools + Memory + Reasoning)

Now let's build a proper agent that can:
- ✅ Look up actual order information (Tool Use)
- ✅ Remember conversation context (Memory)
- ✅ Decide what action to take (Reasoning)

In [12]:
# Simulated database (in real-world, this would be your actual database)
ORDERS_DB = {
    "12345": {
        "customer": "Sarah Johnson",
        "status": "Shipped",
        "items": ["Wireless Headphones", "Phone Case"],
        "tracking": "TRK-789456123",
        "delivery_date": "June 26, 2026"
    },
    "67890": {
        "customer": "Mike Chen",
        "status": "Processing",
        "items": ["Laptop Stand", "USB-C Hub"],
        "tracking": None,
        "delivery_date": "Pending"
    }
}

INVENTORY_DB = {
    "Wireless Headphones": {"stock": 45, "price": 79.99},
    "Phone Case": {"stock": 120, "price": 19.99},
    "Laptop Stand": {"stock": 0, "price": 49.99},  # Out of stock!
    "USB-C Hub": {"stock": 30, "price": 39.99}
}

print("Databases loaded!")

Databases loaded!


In [13]:
from langchain_core.tools import tool

# Define tools that the agent can use

@tool
def lookup_order(order_id: str) -> str:
    """Look up order details by order ID. Returns order status, items, and tracking info."""
    order_id = order_id.replace("#", "").strip()
    if order_id in ORDERS_DB:
        order = ORDERS_DB[order_id]
        return f"""Order #{order_id}:
- Customer: {order['customer']}
- Status: {order['status']}
- Items: {', '.join(order['items'])}
- Tracking: {order['tracking'] or 'Not yet available'}
- Expected Delivery: {order['delivery_date']}"""
    return f"Order #{order_id} not found in our system."

@tool
def check_inventory(product_name: str) -> str:
    """Check if a product is in stock and get its price."""
    for name, info in INVENTORY_DB.items():
        if product_name.lower() in name.lower():
            status = "In Stock" if info['stock'] > 0 else "Out of Stock"
            return f"{name}: {status} ({info['stock']} units) - ${info['price']}"
    return f"Product '{product_name}' not found."

@tool
def initiate_return(order_id: str, reason: str) -> str:
    """Initiate a return request for an order."""
    order_id = order_id.replace("#", "").strip()
    if order_id in ORDERS_DB:
        return f"Return request initiated for Order #{order_id}. Reason: {reason}. Return label will be emailed within 24 hours."
    return f"Cannot initiate return - Order #{order_id} not found."

tools = [lookup_order, check_inventory, initiate_return]
print("Tools defined:", [t.name for t in tools])

Tools defined: ['lookup_order', 'check_inventory', 'initiate_return']


In [14]:
from langchain.agents import create_agent

# Create the Agentic AI bot with tools and memory

system_prompt = """You are a helpful customer service agent for ShopEasy e-commerce.
    
You have access to tools to:
- Look up order status and tracking
- Check product inventory
- Initiate returns

Always use the appropriate tool to get real information. Be friendly and helpful.
If you need to look something up, do it before responding."""

# Create the agent using LangChain 1.x API
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt
)

# Memory store - now uses HumanMessage and AIMessage format
chat_history = []

def agentic_support_bot(user_input: str) -> str:
    """Agentic bot with tools and memory"""
    global chat_history
    
    # Build messages list: system + chat history + current input
    messages = chat_history + [HumanMessage(content=user_input)]
    
    # Invoke the agent with messages
    result = agent.invoke({"messages": messages})
    
    # Extract the response (it's the last message in the result)
    response_text = result["messages"][-1].content
    
    # Update memory
    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=response_text))
    
    return response_text

print("Agentic bot ready!")

Agentic bot ready!


In [15]:
# Test 1: Order lookup (Tool Use)
print("=" * 60)
print("Customer: What is the status of my order #12345?")
print("=" * 60)
response = agentic_support_bot("What is the status of my order #12345?")
print("\nAgent Response:")
print(response)

Customer: What is the status of my order #12345?

Agent Response:
Great! Here's the status of your order #12345:

**Order Status: Shipped** ✓

**Items in your order:**
- Wireless Headphones
- Phone Case

**Tracking Information:** TRK-789456123

**Expected Delivery Date:** June 26, 2026

Your order is on its way! You can use the tracking number to monitor your package's progress. Is there anything else you'd like to know about your order?


### ✅ The agent actually looked up the real order data!

In [16]:
# Test 2: Memory - Follow-up question
print("=" * 60)
print("Customer: When will it arrive?")
print("=" * 60)
response = agentic_support_bot("When will it arrive?")
print("\nAgent Response:")
print(response)

Customer: When will it arrive?

Agent Response:
Based on the tracking information for your order, your package is expected to arrive on **June 26, 2026**.

You can track your shipment in real-time using your tracking number **TRK-789456123** to get more detailed delivery updates and an estimated delivery window.

Is there anything else I can help you with?


### ✅ The agent remembered we were talking about order #12345!

In [18]:
# Test 3: Multi-step reasoning - Complex request
print("=" * 60)
print("Customer: I want to return the headphones. They don't fit well.")
print("=" * 60)
response = agentic_support_bot("I want to return the headphones. They don't fit well.")
print("\nAgent Response:")
print(response)

Customer: I want to return the headphones. They don't fit well.

Agent Response:
Perfect! Your return request has been initiated for order #12345. Here's what happens next:

✓ **Return request approved** for the Wireless Headphones
- **Reason:** Don't fit well
- **Return label:** Will be emailed to you within 24 hours

Once you receive the return label, simply pack up the headphones, print the label, and drop it off at the carrier location. Once we receive and process your return, you'll get a refund.

Is there anything else I can help you with?


### ✅ The agent:
1. Remembered we were discussing order #12345
2. Initiated the return with the reason provided
3. Took real action!

---

##  Your Turn: Practice Exercises

### Exercise 1.1: Test the Inventory Tool
Ask the agent about product availability.

In [21]:
# TODO: Ask the agent if the Laptop Stand is in stock
# Hint: The agent should use the check_inventory tool

response = agentic_support_bot("Is the Laptop Stand available?")
print(response)

Unfortunately, the **Laptop Stand is currently out of stock**. 

However, I have good news - we do have it listed in our system at **$49.99**, which means it's an item we carry and it should be back in stock soon. 

Would you like me to help you with anything else, or would you be interested in a similar product?


### Exercise 1.2: Add a New Tool

Create a tool that allows the agent to apply a discount code.

In [22]:
DISCOUNT_CODES = {
    "SAVE10": {"discount": 10, "min_order": 50},
    "WELCOME20": {"discount": 20, "min_order": 100},
}
 
@tool
def validate_discount_code(code: str) -> str:
    """Validate a discount code and return the discount details."""
    # Normalize the input (strip spaces and uppercase) to ensure robust matching
    cleaned_code = code.strip().upper()
    
    # 1. Check if code exists in DISCOUNT_CODES
    if cleaned_code in DISCOUNT_CODES:
        details = DISCOUNT_CODES[cleaned_code]
        # 2. Return discount details as a string for the agent to read
        return f"Valid code! '{cleaned_code}' provides a {details['discount']}% discount with a minimum order value of {details['min_order']}."
    else:
        # 2. Return "Invalid code" message
        return f"Invalid code: '{code}'. This discount code does not exist."

In [26]:
tools = [check_inventory, validate_discount_code]
 
# 2. Re-create the underlying agent using the correct function and parameters
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt
)

In [27]:
response = agentic_support_bot("Can I use the discount code SAVE10 on my order?")
print(response)

Great news! ✓ **The code SAVE10 is valid!**

**Discount Details:**
- **Discount:** 10% off
- **Minimum Order Value:** $50

As long as your order total is $50 or more, you can apply this code at checkout to receive 10% off. Is there anything else I can help you with?


### Exercise 1.3: Compare Side-by-Side

Fill in the comparison table based on what you observed:

| Feature | GenAI Bot | Agentic Bot |
|---------|-----------|-------------|
| Can look up real order data? | ❌ No | ✅ Yes |
| Remembers previous messages? | ❌ No | ✅ Yes |
| Can take actions (returns)? | ❌ No | ✅ Yes |
| Decides what tool to use? | ❌ No | ✅ Yes |
| Provides accurate info? | ❌ No | ✅ Yes |